<a href="https://colab.research.google.com/github/THEJoshinator20/ST-554-Project1-Template/blob/main/Task1/Task1_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Alana Pooler
<br>
ST 554 Project 1

# Task 1: Prediction of C6H6(GT)

This task involves writing two gradient descent type algorithms to find the optimal constant to use for squared error loss (ends up being the sample mean) and to find the optimal intercept and slope from a simple linear regression model.

## Load and clean data set

In [ ]:
# install ucimlrepo
!pip install ucimlrepo

In [8]:
# import libraries
import ucimlrepo as uci
import numpy as np
import pandas as pd

In [43]:
air_quality = uci.fetch_ucirepo(id=360)
# view data set
air_quality = air_quality.data.features
air_quality.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


Remove any observations where the C6H6(GT) or CO(GT) are -200 as these represent missing values

In [44]:
air_quality_clean = air_quality[(air_quality['C6H6(GT)'] != -200) & (air_quality['CO(GT)'] != -200)]

Since we will mainly be using C6H6(GT) as y and PT08.S1(CO) as x, rename these variables to be easier to reference

In [45]:
air_quality_clean = air_quality_clean.rename(columns = {'C6H6(GT)': 'y', 'PT08.S1(CO)': 'x'})

## Grid Search Algorithm: Just y

Implement a grid search to find the optimal value of c based off the data set.

 Create function to calculate root mean squared error

In [27]:
def rmse(y: pd.Series, c: float) -> float:
  """
  Calculates Root Mean Squared Error (RMSE) for a given value of c.
  """
  # calculate and return RMSE
  return np.sqrt(np.mean((y - c) ** 2))

Write function to find optimal c given y variable:


*   Find the first and third quartiles of C6H6(GT) to determine reasonable values for grid search
*   Use a list comprehensive to loop over the grid of c values, finding the RMSE for each value of c.
*   Determine which value of c gives the optimal (smallest) RMSE.
*   Report that as the prediction





In [29]:
def find_optimal_c(y: pd.Series) -> float:
  """
  Function to find the optimal value of c for a given column.
  """
  # find first and third quartile
  q1, q3 = np.percentile(y, [25, 75])

  # create grid using quartiles
  grid = np.linspace(q1, q3, 100)

  # calculate RMSE for each value in grid
  rmse_values = [rmse(y, c) for c in grid]

  # output optimal value of c
  return grid[np.argmin(rmse_values)]

Test function on C6H6(GT)

According to calculus, the optimal value of c should be the mean of y. The mean of C6H6(GT) is 10.2757, so the grid search algorithm did a pretty good job.

In [60]:
c = find_optimal_c(air_quality_clean['y'])
print(f"Optimal c value = {round(c, 4)}")
print(f"Mean value of C6H6(GT) = {round(air_quality_clean['y'].mean(), 4)}")

Optimal c value = 10.2828
Mean value of C6H6(GT) = 10.2757


Test function on PT08.S1(CO) to make sure algorithm generalizes

Again, the optimal value of c found by the grid search algorithm is very close to the mean value of PT08.S1(CO), so we can conclude that the algorithm generalizes well.

In [61]:
c = find_optimal_c(air_quality_clean['x'])
print(f"Optimal c value = {round(c, 4)}")
print(f"Mean value of PT08.S1(CO) = {round(air_quality_clean['x'].mean(), 4)}")

Optimal c value = 1109.6364
Mean value of PT08.S1(CO) = 1110.5807


## Grid search algorithm: x and y

Implement the grid search to find the optimal pair of values for b0 and b1 using PT08.S1(CO) as your x variable and C6H6(GT) as your y variable.

Function to calculate RMSE:

In [76]:
def rmse_xy(
    x: pd.Series,
    y: pd.Series,
    b0: float,
    b1: float
) -> float:
  """
  Calculates Root Mean Squared Error (RMSE) for a given value of x, y, b0, and b1.
  """
  # calculate and return RMSE
  return np.sqrt(np.mean((y - b0 - b1*x)**2))

Function to find optimal b0 and b1 values:


*   Populate a grid of b0 and b1 values to consider
*   Calulcate RMSE using x, y, b0, and b1
*   Report the optimal b0 and b1 combination based on RMSE



In [77]:
def find_optimal_betas(x: pd.Series, y: pd.Series):
  """
  Finds optimal values of b0 and b1 for a given x and y variable.
  """

  # create b0 and b1 values
  b0_vals, b1_vals = np.arange(-25, -15, 0.1), np.arange(-5, 5, 0.01)

  # create grid of b0 and b1 values
  grid = [(b0, b1) for b0 in b0_vals for b1 in b1_vals]

  # calculate RMSE for each value of b0 and b1
  rmse_values = [rmse_xy(x, y, b0, b1) for b0, b1 in grid]

  # find best values of b0 and b1
  best_b0, best_b1 = grid[np.argmin(rmse_values)]

  return best_b0, best_b1


Test with x = PT08.S1(CO) and y = C6H6(GT)

In [78]:
b0, b1 = find_optimal_betas(air_quality_clean['x'], air_quality_clean['y'])
print(f"Optimal b0 = {round(b0, 4)}, optimal b1 = {round(b1, 4)}")

Optimal b0 = -23.0, optimal b1 = 0.03


Use these values to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

Predicted value for PT08.S1(CO) = 946: 5.38
<br>
Predicted value for PT08.S1(CO) = 1075: 9.25
<br>
Predicted value for PT08.S1(CO) = 1246: 14.38

In [79]:
# use for loop to calculate values of C6H6(GT) using given values of PT08.S1(CO)
for num in [946, 1075, 1246]:
  pred = b0 + b1 * num
  print(round(pred, 4))

5.38
9.25
14.38


## Gradient Descent Algorithm: Just y

Function to calculate RMSE:

In [48]:
def rmse_y(y: pd.Series, c: float) -> float:
  """
  Calculates Root Mean Squared Error (RMSE) for a given value of c.
  """
  # calculate and return RMSE
  return np.sqrt(np.mean((y - c) ** 2))

Function to calculate difference quotient

In [49]:
def diff_quotient_y(y: pd.Series, c: float, delta: float) -> float:
  """
  Calculate difference quotient to approximate the slope of tangent line using a given y, c, and delta.
  """
  # calculate and return difference quotient
  return (rmse_y(y, c + delta) - rmse_y(y, c)) / delta

Gradient descent algorithm

In [50]:
def gradient_descent_y(
    y: pd.Series,
    start_value: float,
    delta: float = 0.001,
    step_size: float = 0.01,
    tolerance: float = 0.0001,
    max_iterations: int = 10000
) -> float:
  """
  Calculate optimal value of c using gradient descent.
  """

  # assign starting value to cur_c
  cur_c = start_value

  for i in range(max_iterations):
    # calculate new_c
    new_c = cur_c - (diff_quotient_y(y, cur_c, delta)) * step_size

    # stop loop if abs(new_c - cur_c) < num_tol
    if abs(new_c - cur_c) < tolerance:
      break

    # otherwise, update cur_c and repeat
    cur_c = new_c

  return cur_c

Test function on C6H6(GT), using 0 as starting value

The result is very similar to the value of c we got from the grid search algorithm, although the grid search algorithm resulted in a c value that was closer to the mean of C6H6(GT).

In [54]:
c = gradient_descent_y(air_quality_clean['y'], start_value = 0)
print(f"Optimal c value for C6H6(GT) = {round(c, 4)}")

Optimal c value for C6H6(GT) = 10.2009


Test function on PT08.S1(CO) to make sure algorithm generalizes, using 1100 as the start value

This value of c is closer to the mean of PT08.S1(CO) than the c value we got from the grid search algorithm. We can conclude that the algorithm does generalize well.

In [55]:
c = gradient_descent_y(air_quality_clean['x'], start_value = 1100, step_size = 0.1)
print(f"Optimal c value for PT08.S1(CO) = {round(c, 4)}")

Optimal c value for PT08.S1(CO) = 1110.3616


## Gradient Descent Algorithm: Using x and y

Implement a gradient descent algorithm to find the optimal b0 and b1 values using C6H6(GT) as y and PT08.S1(CO) as x.

First, create a function to calculate RMSE:

In [ ]:
def rmse_xy(
    x: pd.Series,
    y: pd.Series,
    b0: float,
    b1: float
) -> float:
  # calculate and return RMSE
  return np.sqrt(np.mean((y - b0 - b1*x)**2))